# RAG Fundamentals: Building Your First Retrieval-Augmented Generation System

Welcome to the foundational RAG notebook! In this interactive tutorial, you'll learn:

1. **Understanding Embeddings** - How text becomes vectors
2. **Chunking Strategies** - Optimal text segmentation techniques
3. **Vector Search** - Finding relevant information efficiently
4. **End-to-end RAG** - Putting it all together

## Learning Objectives
By the end of this notebook, you will:
- Understand how embeddings represent text semantically
- Compare different chunking strategies and their trade-offs
- Implement vector similarity search from scratch
- Build a complete RAG pipeline with evaluation metrics

---

In [ ]:
# Install required packages
!pip install -q openai langchain sentence-transformers numpy scikit-learn matplotlib seaborn plotly pandas tiktoken

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer
import tiktoken
import re
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

## 1. Understanding Embeddings: Text to Vectors

Embeddings are the foundation of RAG systems. They transform text into high-dimensional vectors that capture semantic meaning.

In [ ]:
# Sample documents for our RAG system
sample_documents = [
    "Machine learning is a subset of artificial intelligence that focuses on algorithms that learn from data.",
    "Deep learning uses neural networks with multiple layers to model complex patterns in data.",
    "Natural language processing enables computers to understand and generate human language.",
    "Computer vision allows machines to interpret and understand visual information from images.",
    "Reinforcement learning trains agents to make decisions through interaction with an environment.",
    "Data science combines statistics, programming, and domain expertise to extract insights from data.",
    "Cloud computing provides on-demand access to computing resources over the internet.",
    "Cybersecurity protects digital systems and data from unauthorized access and attacks.",
    "Blockchain is a distributed ledger technology that ensures secure and transparent transactions.",
    "Quantum computing leverages quantum mechanical properties to perform certain calculations exponentially faster."
]

print(f"Sample dataset: {len(sample_documents)} documents")
for i, doc in enumerate(sample_documents, 1):
    print(f"{i:2d}. {doc[:60]}...")

In [ ]:
# Initialize embedding model
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for our documents
document_embeddings = embedding_model.encode(sample_documents)

print(f"Embedding dimensions: {document_embeddings.shape[1]}")
print(f"Number of document embeddings: {document_embeddings.shape[0]}")
print(f"\nFirst document embedding (first 10 dimensions):")
print(document_embeddings[0][:10])

### Visualizing Embeddings in 2D Space

In [ ]:
# Reduce embeddings to 2D for visualization using PCA
pca = PCA(n_components=2, random_state=42)
embeddings_2d = pca.fit_transform(document_embeddings)

# Create interactive plot
fig = go.Figure()

# Add points for each document
for i, (x, y) in enumerate(embeddings_2d):
    fig.add_trace(go.Scatter(
        x=[x], y=[y],
        mode='markers+text',
        marker=dict(size=12, opacity=0.7),
        text=f"Doc {i+1}",
        textposition="top center",
        hovertext=sample_documents[i],
        hoverinfo="text",
        name=f"Document {i+1}"
    ))

fig.update_layout(
    title="Document Embeddings in 2D Space (PCA Projection)",
    xaxis_title="PC1",
    yaxis_title="PC2",
    showlegend=False,
    height=600
)

fig.show()

print(f"PCA explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.3f}")

### Similarity Heatmap

In [ ]:
# Calculate cosine similarity matrix
similarity_matrix = cosine_similarity(document_embeddings)

# Create heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(similarity_matrix, 
            annot=True, 
            fmt='.3f',
            cmap='viridis',
            xticklabels=[f"Doc {i+1}" for i in range(len(sample_documents))],
            yticklabels=[f"Doc {i+1}" for i in range(len(sample_documents))])
plt.title('Document Similarity Matrix (Cosine Similarity)')
plt.tight_layout()
plt.show()

# Find most similar document pairs
similarity_pairs = []
for i in range(len(sample_documents)):
    for j in range(i+1, len(sample_documents)):
        similarity_pairs.append({
            'doc1': i+1,
            'doc2': j+1,
            'similarity': similarity_matrix[i][j],
            'text1': sample_documents[i][:50] + '...',
            'text2': sample_documents[j][:50] + '...'
        })

similarity_df = pd.DataFrame(similarity_pairs)
print("\nTop 5 Most Similar Document Pairs:")
print(similarity_df.nlargest(5, 'similarity')[['doc1', 'doc2', 'similarity', 'text1', 'text2']])

## 2. Chunking Strategies: Breaking Down Text Optimally

Different chunking strategies have different trade-offs. Let's compare them!

In [ ]:
# Long sample text for chunking demonstration
long_text = """
Artificial Intelligence (AI) represents one of the most transformative technologies of our time. 
At its core, AI refers to the development of computer systems that can perform tasks typically 
requiring human intelligence, such as learning, reasoning, problem-solving, perception, and 
language understanding.

Machine Learning (ML), a subset of AI, has emerged as a particularly powerful approach. 
Instead of being explicitly programmed for every task, ML systems learn patterns from data 
and make predictions or decisions based on that learning. This paradigm shift has enabled 
remarkable advances in areas like image recognition, natural language processing, and 
autonomous systems.

Deep Learning, a specialized branch of ML, uses artificial neural networks with multiple 
layers to model and understand complex patterns. These networks, inspired by the structure 
of the human brain, have proven exceptionally effective at tasks like image classification, 
speech recognition, and language translation. The "deep" in deep learning refers to the 
many layers these networks contain.

Natural Language Processing (NLP) focuses specifically on enabling computers to understand, 
interpret, and generate human language. Recent advances in NLP, particularly with large 
language models like GPT and BERT, have revolutionized how machines interact with human 
language, enabling applications from chatbots to automated translation services.

The practical applications of AI are vast and growing. In healthcare, AI assists in 
diagnostic imaging and drug discovery. In transportation, it powers autonomous vehicles. 
In finance, it enables fraud detection and algorithmic trading. In entertainment, it 
personalizes content recommendations. The potential seems limitless.
""".strip()

print(f"Text length: {len(long_text)} characters")
print(f"Word count: {len(long_text.split())} words")

In [ ]:
class ChunkingStrategy:
    """Compare different text chunking approaches"""
    
    @staticmethod
    def fixed_size_chunking(text: str, chunk_size: int = 500, overlap: int = 50) -> List[str]:
        """Split text into fixed-size chunks with optional overlap"""
        chunks = []
        start = 0
        while start < len(text):
            end = start + chunk_size
            chunk = text[start:end]
            chunks.append(chunk.strip())
            start = end - overlap
        return [chunk for chunk in chunks if chunk.strip()]
    
    @staticmethod
    def sentence_chunking(text: str, sentences_per_chunk: int = 3) -> List[str]:
        """Split text by sentences, grouping N sentences per chunk"""
        # Simple sentence splitting (could use spaCy or NLTK for better accuracy)
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        for i in range(0, len(sentences), sentences_per_chunk):
            chunk = '. '.join(sentences[i:i + sentences_per_chunk])
            if chunk:
                chunks.append(chunk + '.')
        return chunks
    
    @staticmethod
    def paragraph_chunking(text: str) -> List[str]:
        """Split text by paragraphs"""
        paragraphs = text.split('\n\n')
        return [p.strip() for p in paragraphs if p.strip()]
    
    @staticmethod
    def semantic_chunking(text: str, model, similarity_threshold: float = 0.7) -> List[str]:
        """Group sentences based on semantic similarity"""
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        if len(sentences) <= 1:
            return sentences
        
        # Encode sentences
        embeddings = model.encode(sentences)
        
        chunks = [sentences[0]]
        current_chunk_embedding = embeddings[0].reshape(1, -1)
        
        for i in range(1, len(sentences)):
            sentence_embedding = embeddings[i].reshape(1, -1)
            similarity = cosine_similarity(current_chunk_embedding, sentence_embedding)[0][0]
            
            if similarity > similarity_threshold:
                # Add to current chunk
                chunks[-1] += '. ' + sentences[i]
                # Update chunk embedding (average)
                current_chunk_embedding = np.mean([current_chunk_embedding[0], sentence_embedding[0]], axis=0).reshape(1, -1)
            else:
                # Start new chunk
                chunks.append(sentences[i])
                current_chunk_embedding = sentence_embedding
        
        return [chunk + '.' for chunk in chunks]

In [ ]:
# Apply different chunking strategies
chunking_results = {
    'Fixed Size (300 chars)': ChunkingStrategy.fixed_size_chunking(long_text, chunk_size=300, overlap=50),
    'Sentence-based (2 sentences)': ChunkingStrategy.sentence_chunking(long_text, sentences_per_chunk=2),
    'Paragraph-based': ChunkingStrategy.paragraph_chunking(long_text),
    'Semantic': ChunkingStrategy.semantic_chunking(long_text, embedding_model, similarity_threshold=0.6)
}

# Display results
for strategy, chunks in chunking_results.items():
    print(f"\n{'='*50}")
    print(f"Strategy: {strategy}")
    print(f"Number of chunks: {len(chunks)}")
    print(f"{'='*50}")
    
    for i, chunk in enumerate(chunks, 1):
        print(f"\nChunk {i} ({len(chunk)} chars):")
        print(chunk[:200] + ('...' if len(chunk) > 200 else ''))


### Chunking Strategy Comparison

In [ ]:
# Analyze chunking strategies
chunk_analysis = []

for strategy, chunks in chunking_results.items():
    chunk_lengths = [len(chunk) for chunk in chunks]
    
    analysis = {
        'Strategy': strategy,
        'Num Chunks': len(chunks),
        'Avg Length': np.mean(chunk_lengths),
        'Std Length': np.std(chunk_lengths),
        'Min Length': min(chunk_lengths),
        'Max Length': max(chunk_lengths)
    }
    chunk_analysis.append(analysis)

analysis_df = pd.DataFrame(chunk_analysis)
print("Chunking Strategy Comparison:")
print(analysis_df.round(2))

# Visualize chunk lengths
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
axes = axes.ravel()

for i, (strategy, chunks) in enumerate(chunking_results.items()):
    chunk_lengths = [len(chunk) for chunk in chunks]
    axes[i].bar(range(len(chunk_lengths)), chunk_lengths, alpha=0.7)
    axes[i].set_title(f'{strategy}\n({len(chunks)} chunks)')
    axes[i].set_xlabel('Chunk Index')
    axes[i].set_ylabel('Character Count')

plt.tight_layout()
plt.show()

## 3. Vector Search: Finding Relevant Information

Now let's implement vector search to find the most relevant chunks for a query.

In [ ]:
class VectorSearch:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, documents: List[str]):
        """Add documents to the search index"""
        self.documents.extend(documents)
        new_embeddings = self.embedding_model.encode(documents)
        
        if self.embeddings is None:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack([self.embeddings, new_embeddings])
    
    def search(self, query: str, k: int = 5) -> List[Tuple[str, float]]:
        """Search for top-k most similar documents"""
        if self.embeddings is None:
            return []
        
        # Encode query
        query_embedding = self.embedding_model.encode([query])
        
        # Calculate similarities
        similarities = cosine_similarity(query_embedding, self.embeddings)[0]
        
        # Get top-k results
        top_indices = np.argsort(similarities)[::-1][:k]
        
        results = []
        for idx in top_indices:
            results.append((self.documents[idx], similarities[idx]))
        
        return results
    
    def visualize_search(self, query: str, k: int = 5):
        """Visualize search results with similarity scores"""
        results = self.search(query, k)
        
        if not results:
            print("No results found")
            return
        
        print(f"Query: '{query}'\n")
        print(f"Top {k} Results:")
        print("="*80)
        
        documents, scores = zip(*results)
        
        # Create bar chart
        plt.figure(figsize=(12, 6))
        bars = plt.barh(range(len(scores)), scores, alpha=0.7)
        plt.xlabel('Similarity Score')
        plt.ylabel('Documents')
        plt.title(f'Top {k} Search Results for: "{query}"')
        
        # Add labels
        labels = [f"Doc {i+1}" for i in range(len(scores))]
        plt.yticks(range(len(scores)), labels)
        
        # Add score annotations
        for i, (bar, score) in enumerate(zip(bars, scores)):
            plt.text(bar.get_width() - 0.01, bar.get_y() + bar.get_height()/2, 
                    f'{score:.3f}', ha='right', va='center')
        
        plt.tight_layout()
        plt.show()
        
        # Print detailed results
        for i, (doc, score) in enumerate(results, 1):
            print(f"\n{i}. Similarity: {score:.4f}")
            print(f"   {doc[:150]}{'...' if len(doc) > 150 else ''}")
        
        return results

In [ ]:
# Create vector search index with our chunked documents
vector_search = VectorSearch(embedding_model)

# Use paragraph-based chunks for this demo
chunks = chunking_results['Paragraph-based']
vector_search.add_documents(chunks)

print(f"Added {len(chunks)} chunks to vector search index")

In [ ]:
# Test queries
test_queries = [
    "What is machine learning?",
    "How do neural networks work?",
    "Applications of AI in healthcare",
    "What is natural language processing?"
]

for query in test_queries:
    print("\n" + "="*100)
    results = vector_search.visualize_search(query, k=3)
    print("\n" + "-"*50 + "\n")

## 4. End-to-End RAG Pipeline

Now let's build a complete RAG system that combines retrieval with generation.

In [ ]:
class SimpleRAGPipeline:
    def __init__(self, embedding_model):
        self.vector_search = VectorSearch(embedding_model)
        self.embedding_model = embedding_model
    
    def add_documents(self, documents: List[str]):
        """Add documents to the RAG system"""
        self.vector_search.add_documents(documents)
    
    def generate_response(self, query: str, k: int = 3) -> Dict:
        """Generate response using retrieved context"""
        # Retrieve relevant documents
        retrieved_docs = self.vector_search.search(query, k=k)
        
        if not retrieved_docs:
            return {
                'query': query,
                'retrieved_docs': [],
                'response': "No relevant documents found.",
                'confidence': 0.0
            }
        
        # Extract documents and scores
        docs, scores = zip(*retrieved_docs)
        
        # Create context from retrieved documents
        context = "\n\n".join(docs)
        
        # Simple template-based response (in practice, you'd use an LLM here)
        response = self._generate_template_response(query, context, scores)
        
        return {
            'query': query,
            'retrieved_docs': retrieved_docs,
            'context': context,
            'response': response,
            'avg_confidence': np.mean(scores)
        }
    
    def _generate_template_response(self, query: str, context: str, scores: List[float]) -> str:
        """Generate a template-based response (placeholder for LLM)"""
        avg_score = np.mean(scores)
        
        response = f"Based on the available information (confidence: {avg_score:.3f}):\n\n"
        
        if avg_score > 0.7:
            response += "I found highly relevant information to answer your question:\n"
        elif avg_score > 0.5:
            response += "I found moderately relevant information that may help:\n"
        else:
            response += "I found some potentially related information:\n"
        
        # Extract key sentences from context
        sentences = context.split('. ')
        key_sentences = sentences[:3]  # Take first 3 sentences
        
        response += "\n".join([f"• {sentence.strip()}" for sentence in key_sentences if sentence.strip()])
        
        return response

In [ ]:
# Initialize RAG pipeline
rag_pipeline = SimpleRAGPipeline(embedding_model)

# Add documents (using our paragraph chunks)
rag_pipeline.add_documents(chunks)

print("RAG Pipeline initialized successfully!")

In [ ]:
# Test the RAG pipeline
test_questions = [
    "What is the difference between AI and machine learning?",
    "How does deep learning work?",
    "What are some practical applications of AI?",
    "Explain natural language processing"
]

for question in test_questions:
    print("\n" + "="*100)
    print(f"QUESTION: {question}")
    print("="*100)
    
    result = rag_pipeline.generate_response(question, k=2)
    
    print(f"\nRESPONSE (Confidence: {result['avg_confidence']:.3f}):")
    print("-" * 50)
    print(result['response'])
    
    print(f"\nRETRIEVED DOCUMENTS:")
    print("-" * 50)
    for i, (doc, score) in enumerate(result['retrieved_docs'], 1):
        print(f"{i}. Score: {score:.3f}")
        print(f"   {doc[:200]}{'...' if len(doc) > 200 else ''}")
        print()

## 5. RAG Evaluation Metrics

Let's implement some basic metrics to evaluate our RAG system performance.

In [ ]:
class RAGEvaluator:
    def __init__(self, embedding_model):
        self.embedding_model = embedding_model
    
    def calculate_retrieval_metrics(self, query: str, retrieved_docs: List[str], 
                                  ground_truth_docs: List[str]) -> Dict:
        """Calculate precision, recall, and F1 for retrieval"""
        if not ground_truth_docs:
            return {'precision': 0, 'recall': 0, 'f1': 0}
        
        # Simple string matching for ground truth (in practice, use semantic similarity)
        relevant_retrieved = 0
        for retrieved in retrieved_docs:
            for ground_truth in ground_truth_docs:
                # Use cosine similarity to determine relevance
                emb1 = self.embedding_model.encode([retrieved])
                emb2 = self.embedding_model.encode([ground_truth])
                similarity = cosine_similarity(emb1, emb2)[0][0]
                if similarity > 0.8:  # Threshold for relevance
                    relevant_retrieved += 1
                    break
        
        precision = relevant_retrieved / len(retrieved_docs) if retrieved_docs else 0
        recall = relevant_retrieved / len(ground_truth_docs) if ground_truth_docs else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
        
        return {
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'relevant_retrieved': relevant_retrieved
        }
    
    def calculate_response_relevance(self, query: str, response: str) -> float:
        """Calculate semantic similarity between query and response"""
        query_emb = self.embedding_model.encode([query])
        response_emb = self.embedding_model.encode([response])
        return cosine_similarity(query_emb, response_emb)[0][0]
    
    def evaluate_rag_pipeline(self, test_cases: List[Dict]) -> pd.DataFrame:
        """Evaluate RAG pipeline on multiple test cases"""
        results = []
        
        for case in test_cases:
            query = case['query']
            ground_truth = case.get('ground_truth_docs', [])
            
            # Get RAG response
            rag_result = rag_pipeline.generate_response(query)
            retrieved_docs = [doc for doc, _ in rag_result['retrieved_docs']]
            
            # Calculate metrics
            retrieval_metrics = self.calculate_retrieval_metrics(query, retrieved_docs, ground_truth)
            response_relevance = self.calculate_response_relevance(query, rag_result['response'])
            
            results.append({
                'query': query,
                'precision': retrieval_metrics['precision'],
                'recall': retrieval_metrics['recall'],
                'f1': retrieval_metrics['f1'],
                'response_relevance': response_relevance,
                'avg_retrieval_score': rag_result['avg_confidence'],
                'num_retrieved': len(retrieved_docs)
            })
        
        return pd.DataFrame(results)

In [ ]:
# Create test cases with ground truth
test_cases = [
    {
        'query': 'What is machine learning?',
        'ground_truth_docs': [
            "Machine Learning (ML), a subset of AI, has emerged as a particularly powerful approach. Instead of being explicitly programmed for every task, ML systems learn patterns from data and make predictions or decisions based on that learning."
        ]
    },
    {
        'query': 'How do neural networks work?',
        'ground_truth_docs': [
            "Deep Learning, a specialized branch of ML, uses artificial neural networks with multiple layers to model and understand complex patterns. These networks, inspired by the structure of the human brain, have proven exceptionally effective at tasks like image classification, speech recognition, and language translation."
        ]
    },
    {
        'query': 'What are AI applications in healthcare?',
        'ground_truth_docs': [
            "In healthcare, AI assists in diagnostic imaging and drug discovery."
        ]
    }
]

# Evaluate the RAG pipeline
evaluator = RAGEvaluator(embedding_model)
evaluation_results = evaluator.evaluate_rag_pipeline(test_cases)

print("RAG Pipeline Evaluation Results:")
print("=" * 50)
print(evaluation_results.round(3))

# Summary statistics
print("\nSummary Statistics:")
print("=" * 50)
print(evaluation_results.describe().round(3))

In [ ]:
# Visualize evaluation metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Precision, Recall, F1
metrics = ['precision', 'recall', 'f1']
x = np.arange(len(evaluation_results))
width = 0.25

for i, metric in enumerate(metrics):
    axes[0, 0].bar(x + i*width, evaluation_results[metric], width, label=metric.capitalize())

axes[0, 0].set_xlabel('Test Case')
axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Retrieval Metrics by Test Case')
axes[0, 0].set_xticks(x + width)
axes[0, 0].set_xticklabels([f'Q{i+1}' for i in range(len(evaluation_results))])
axes[0, 0].legend()
axes[0, 0].set_ylim(0, 1)

# Response relevance
axes[0, 1].bar(range(len(evaluation_results)), evaluation_results['response_relevance'], alpha=0.7)
axes[0, 1].set_xlabel('Test Case')
axes[0, 1].set_ylabel('Relevance Score')
axes[0, 1].set_title('Response Relevance')
axes[0, 1].set_xticks(range(len(evaluation_results)))
axes[0, 1].set_xticklabels([f'Q{i+1}' for i in range(len(evaluation_results))])

# Average retrieval scores
axes[1, 0].bar(range(len(evaluation_results)), evaluation_results['avg_retrieval_score'], alpha=0.7, color='green')
axes[1, 0].set_xlabel('Test Case')
axes[1, 0].set_ylabel('Average Score')
axes[1, 0].set_title('Average Retrieval Confidence')
axes[1, 0].set_xticks(range(len(evaluation_results)))
axes[1, 0].set_xticklabels([f'Q{i+1}' for i in range(len(evaluation_results))])

# Correlation heatmap
corr_matrix = evaluation_results[['precision', 'recall', 'f1', 'response_relevance', 'avg_retrieval_score']].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, ax=axes[1, 1])
axes[1, 1].set_title('Metric Correlations')

plt.tight_layout()
plt.show()

## 6. Interactive Exercise: Build Your Own RAG System

Now it's your turn! Complete the following exercise to reinforce your learning.

In [ ]:
# Exercise: Implement a RAG system for a specific domain

# Sample domain-specific documents (cooking recipes)
cooking_documents = [
    "To make pasta, boil water with salt, add pasta, and cook for 8-12 minutes depending on type.",
    "For a perfect risotto, use arborio rice and add warm broth gradually while stirring constantly.",
    "When baking bread, knead the dough until smooth and let it rise in a warm place for 1 hour.",
    "To grill chicken, marinate for at least 30 minutes and cook on medium-high heat for 6-8 minutes per side.",
    "For chocolate chip cookies, cream butter and sugar, add eggs and vanilla, then fold in flour and chocolate chips.",
    "When making soup, start with a good base like sautéed onions, garlic, and celery (mirepoix).",
    "To properly season food, taste as you go and adjust salt, pepper, and other seasonings accordingly.",
    "For tender vegetables, don't overcook them - they should retain some bite and vibrant color."
]

# TODO: Complete this exercise
# 1. Create a new RAG pipeline for cooking
# 2. Add the cooking documents
# 3. Test it with cooking-related questions
# 4. Evaluate the results

print("Exercise: Build a cooking RAG system")
print("Your task: Complete the implementation below")
print(f"You have {len(cooking_documents)} cooking documents to work with")

# Your code here:
cooking_rag = SimpleRAGPipeline(embedding_model)
cooking_rag.add_documents(cooking_documents)

# Test questions
cooking_questions = [
    "How do I cook pasta properly?",
    "What's the secret to good risotto?",
    "How should I grill chicken?",
    "Tips for making cookies?"
]

print("\nTesting cooking RAG system:")
for question in cooking_questions:
    result = cooking_rag.generate_response(question, k=2)
    print(f"\nQ: {question}")
    print(f"A: {result['response']}")
    print(f"Confidence: {result['avg_confidence']:.3f}")

## Summary and Key Takeaways

🎉 **Congratulations!** You've successfully built and evaluated a complete RAG system.

### What You've Learned:

1. **Embeddings Fundamentals**
   - How text transforms into meaningful vector representations
   - Semantic similarity through cosine distance
   - Visualization techniques for high-dimensional embeddings

2. **Chunking Strategies**
   - Fixed-size vs. semantic chunking trade-offs
   - Impact of chunk size on retrieval quality
   - When to use different strategies

3. **Vector Search Implementation**
   - Building search indices from scratch
   - Similarity scoring and ranking
   - Performance visualization

4. **End-to-End RAG Pipeline**
   - Integrating retrieval with generation
   - Context formatting and response synthesis
   - Confidence scoring

5. **Evaluation Metrics**
   - Retrieval quality (precision, recall, F1)
   - Response relevance scoring
   - A/B testing framework

### Next Steps:
- Experiment with different embedding models
- Try advanced chunking techniques (recursive, semantic)
- Implement reranking for better results
- Add query expansion and preprocessing
- Explore hybrid search (vector + keyword)

### Production Considerations:
- Vector database integration (Pinecone, Weaviate, Chroma)
- Caching strategies for embeddings
- Batch processing for large document sets
- Monitoring and observability
- Cost optimization techniques

Ready to move on to advanced prompt engineering? Check out notebook **02-prompt-engineering.ipynb**! 🚀